# Support Ticket Classification and Response Generation with Runnables

In [ ]:
# -----------------------------
# Imports
# -----------------------------
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableAssign
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import Literal

# -----------------------------
# Load environment variables
# -----------------------------
load_dotenv()

# -----------------------------
# Initialize model
# -----------------------------
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER")
)


parser = StrOutputParser()

In [ ]:
# -----------------------------
# Pydantic models
# -----------------------------
class ClassificationResult(BaseModel):
    category: Literal["billing", "technical", "account"] = Field(
        description="Support ticket category"
    )


class UserInfo(BaseModel):
    name: str
    email: str
    subscription_level: Literal["free", "premium", "enterprise"]
    account_number: str

In [ ]:
# -----------------------------
# Step 1: Classification Prompt
# -----------------------------
classification_prompt = PromptTemplate(
    template="""
Classify the following support message into one category:
billing, technical, or account.

Message:
{message}
""",
    input_variables=["message"]
)

classification_chain = (
    classification_prompt
    | model.with_structured_output(ClassificationResult)
)

In [ ]:
# -----------------------------
# Step 2: Extract user info
# -----------------------------
user_info_prompt = PromptTemplate(
    template="""
Extract the following user information from this support ticket.

Message:
{message}

Return:
- name
- email
- subscription_level (free, premium, enterprise)
- account_number
""",
    input_variables=["message"]
)

user_info_chain = (
    user_info_prompt
    | model.with_structured_output(UserInfo)
)

In [ ]:
# -----------------------------
# Category → Support team mapping
# -----------------------------
def get_team_name(category: str) -> str:
    mapping = {
        "billing": "Billing Support Team",
        "technical": "Technical Support Team",
        "account": "Account Support Team"
    }
    return mapping.get(category, "Customer Support Team")


# -----------------------------
# Step 3: Response Prompt
# -----------------------------
response_prompt = PromptTemplate(
    template="""
Customer support category: {category}

Customer information:
Name: {name}
Email: {email}
Subscription: {subscription_level}
Account Number: {account_number}

Customer query:
{query}

Write a short professional support response helping the customer.

Rules:
- Address the customer by name
- Do NOT include an email subject
- Always end the response exactly with:

Best regards,
{team_name}
""",
    input_variables=[
        "category",
        "name",
        "email",
        "subscription_level",
        "account_number",
        "query",
        "team_name"
    ]
)

response_chain = response_prompt | model | parser

In [ ]:
# -----------------------------
# Parallel extraction pipeline
# -----------------------------
pipeline = RunnableParallel(
    classification=classification_chain,
    user=user_info_chain,
    query=RunnablePassthrough()  # keep original input
)

# -----------------------------
# Prepare inputs for response
# ---------------------------------------------------------
# RunnableAssign: Reshapes the data flowing through the pipeline.
# It reads the output from the previous step
# and adds new flattened fields required by the final prompt.
#
# RunnableAssign itself is a runnable, so it can be composed
# directly in the pipeline using the `|` operator.
# -----------------------------
prepare_response_inputs = RunnableAssign({
    "category": lambda x: x["classification"].category,
    "name": lambda x: x["user"].name,
    "email": lambda x: x["user"].email,
    "subscription_level": lambda x: x["user"].subscription_level,
    "account_number": lambda x: x["user"].account_number,
    "query": lambda x: x["query"]["message"],
    "team_name": lambda x: get_team_name(x["classification"].category)
})

In [ ]:
# -----------------------------
# Final chain
# -----------------------------
chain = pipeline | prepare_response_inputs | response_chain

# -----------------------------
# Example run
# -----------------------------
result = chain.invoke({
    "message": """
Hello, my name is John Doe.
My email is john@example.com.
Account number: ACC123.
I was charged twice for my premium subscription on 2023-10-01.
"""
})

print(result)